In [1]:
!pip install polars scikit-learn matplotlib seaborn fastexcel --quiet

In [2]:
import os
import warnings
import polars as pl
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import accuracy_score, classification_report

**INGESTION**

**Fabric Lakehouse Paths**

In [3]:
BRONZE = "/lakehouse/default/Files/bronze"
SILVER = "/lakehouse/default/Files/silver"

In [4]:
import polars as pl

# Read from local Downloads folder
source_xlsx = "/lakehouse/default/Files/bronze/E_Commerce_Dataset.xlsx"

# Read and save as parquet to Bronze in Lakehouse
ecommerce_df = pl.read_excel(source_xlsx)
ecommerce_df.write_parquet(f"{BRONZE}/ecommerce_raw.parquet", compression="snappy")

ecommerce_df.head(3)

CustomerID,Churn,Tenure,PreferredLoginDevice,CityTier,WarehouseToHome,PreferredPaymentMode,Gender,HourSpendOnApp,NumberOfDeviceRegistered,PreferedOrderCat,SatisfactionScore,MaritalStatus,NumberOfAddress,Complain,OrderAmountHikeFromlastYear,CouponUsed,OrderCount,DaySinceLastOrder,CashbackAmount
i64,i64,i64,str,i64,i64,str,str,i64,i64,str,i64,str,i64,i64,i64,i64,i64,i64,f64
50001,1,4,"""Mobile Phone""",3,6,"""Debit Card""","""Female""",3,3,"""Laptop & Accessory""",2,"""Single""",9,1,11,1,1,5,159.93
50002,1,null,"""Phone""",1,8,"""UPI""","""Male""",3,4,"""Mobile""",3,"""Single""",7,1,15,0,1,0,120.9
50003,1,null,"""Phone""",1,30,"""Debit Card""","""Male""",2,4,"""Mobile""",3,"""Single""",6,1,14,0,1,3,120.28


**TRANSFORMATION**

In [5]:
# CLEAN (Bronze → Silver) 

# Read from Bronze Parquet
ecommerce_df = pl.read_parquet(f"{BRONZE}/ecommerce_raw.parquet")

# Standardise column names — lowercase and strip spaces
ecommerce_df = ecommerce_df.rename({
    col: col.strip().lower() for col in ecommerce_df.columns
})

# Drop rows where customerid or churn is null
ecommerce_df = ecommerce_df.drop_nulls(subset=["customerid", "churn"])

# Cast dtypes in one pass
ecommerce_df = ecommerce_df.with_columns([
    # Categoricals
    pl.col("preferredlogindevice").cast(pl.Categorical),
    pl.col("preferredpaymentmode").cast(pl.Categorical),
    pl.col("gender").cast(pl.Categorical),
    pl.col("preferedordercat").cast(pl.Categorical),
    pl.col("maritalstatus").cast(pl.Categorical),
    pl.col("complain").cast(pl.Boolean),
    # Numerics
    pl.col("tenure").cast(pl.Float32),
    pl.col("warehousetohome").cast(pl.Float32),
    pl.col("hourspendonapp").cast(pl.Float32),
    pl.col("numberofdeviceregistered").cast(pl.Int16),
    pl.col("satisfactionscore").cast(pl.Int8),
    pl.col("numberofaddress").cast(pl.Int16),
    pl.col("orderamounthikefromlastyear").cast(pl.Float32),
    pl.col("couponused").cast(pl.Float32),
    pl.col("ordercount").cast(pl.Float32),
    pl.col("daysincelastorder").cast(pl.Float32),
    pl.col("cashbackamount").cast(pl.Float32),
    # Target
    pl.col("churn").cast(pl.Int8),
])

# Input nulls with median
null_cols = [c for c in ecommerce_df.columns if ecommerce_df[c].null_count() > 0]

ecommerce_df = ecommerce_df.with_columns([
    pl.col("tenure").fill_null(pl.col("tenure").median()),
    pl.col("warehousetohome").fill_null(pl.col("warehousetohome").median()),
    pl.col("hourspendonapp").fill_null(pl.col("hourspendonapp").median()),
    pl.col("orderamounthikefromlastyear")
      .fill_null(pl.col("orderamounthikefromlastyear").median()),
    pl.col("couponused").fill_null(pl.col("couponused").median()),
    pl.col("ordercount").fill_null(pl.col("ordercount").median()),
    pl.col("daysincelastorder").fill_null(pl.col("daysincelastorder").median()),
])

# Remove duplicates
before = len(ecommerce_df)
ecommerce_df = ecommerce_df.unique(subset=["customerid"], keep="first")

# Remove logical inconsistencies
before = len(ecommerce_df)
ecommerce_df = ecommerce_df.filter(
    pl.col("tenure") >= 0,
    pl.col("ordercount") >= 0,
    pl.col("daysincelastorder") >= 0,
    pl.col("cashbackamount") >= 0,
)


/tmp/ipykernel_128/897940227.py:15: CategoricalRemappingWarning: Local categoricals have different encodings, expensive re-encoding is done to perform this merge operation. Consider using a StringCache or an Enum type if the categories are known in advance
  ecommerce_df = ecommerce_df.with_columns([


In [6]:
ecommerce_df = ecommerce_df.rename({
    "preferredlogindevice"       : "login_device",
    "warehousetohome"            : "warehouse_to_home",
    "citytier"                   : "city_tier",
    "preferredpaymentmode"       : "payment_mode",
    "hourspendonapp"             : "hours_on_app",
    "numberofdeviceregistered"   : "device_registered",
    "preferedordercat"           : "order_cart",
    "satisfactionscore"          : "satisfaction_score",
    "maritalstatus"              : "marital_status",
    "numberofaddress"            : "number_of_address",
    "orderamounthikefromlastyear": "order_amount_hike_from_last_year",
    "couponused"                 : "coupon_used",
    "ordercount"                 : "order_count",
    "daysincelastorder"          : "day_since_last_order",
    "cashbackamount"             : "cashback_amount",
})


#Write to Silver 
ecommerce_df.write_parquet(f"{SILVER}/ecommerce_cleaned.parquet",
                           compression="snappy")

In [7]:
display(ecommerce_df)

**FEATURED ENGINEERING**

In [8]:
# FEATURE ENGINEERING

# Read from Silver
ecommerce_df = pl.read_parquet(f"{SILVER}/ecommerce_cleaned.parquet")

# Fix categorical columns after reading parquet
cat_cols = ["login_device", "payment_mode", "gender", "order_cart", "marital_status"]
ecommerce_df = ecommerce_df.with_columns([
    pl.col(c).cast(pl.Utf8).cast(pl.Categorical) for c in cat_cols
])

# Feature Engineering
ecommerce_feat = ecommerce_df.with_columns([

    # Engagement features
    pl.when(pl.col("day_since_last_order") > 14)
      .then(pl.lit(2))
      .when(pl.col("day_since_last_order") > 7)
      .then(pl.lit(1))
      .otherwise(pl.lit(0))
      .cast(pl.Int8)
      .alias("recency_risk"),

    # Satisfaction risk band
    pl.when(pl.col("satisfaction_score") <= 2)
      .then(pl.lit("low"))
      .when(pl.col("satisfaction_score") == 3)
      .then(pl.lit("medium"))
      .otherwise(pl.lit("high"))
      .alias("satisfaction_band"),

    # Cashback per order — loyalty proxy
    (pl.col("cashback_amount") /
     (pl.col("order_count") + 1))
      .cast(pl.Float32)
      .alias("cashback_per_order"),

    # Coupon dependency — only buys with coupons?
    (pl.col("coupon_used") /
     (pl.col("order_count") + 1))
      .cast(pl.Float32)
      .alias("coupon_dependency"),

    # Order value growth flag
    pl.when(pl.col("order_amount_hike_from_last_year") > 20)
      .then(pl.lit(True))
      .otherwise(pl.lit(False))
      .alias("high_order_growth"),

    # Tenure segment
    pl.when(pl.col("tenure") < 6)
      .then(pl.lit("new"))
      .when(pl.col("tenure") < 18)
      .then(pl.lit("growing"))
      .otherwise(pl.lit("loyal"))
      .alias("tenure_segment"),

    # Engagement intensity — hours on app per device
    (pl.col("hours_on_app") /
     (pl.col("device_registered") + 1))
      .cast(pl.Float32)
      .alias("app_engagement_intensity"),


        # Risk composite score (0–1)
    (
        ((5 - pl.col("satisfaction_score")) / 4.0) * 0.35
        + (pl.col("day_since_last_order") /
           (pl.col("day_since_last_order").max() + 1)) * 0.30
        + (1 - (pl.col("hours_on_app") /
                (pl.col("hours_on_app").max() + 1))) * 0.20
        + pl.col("complain").cast(pl.Float32) * 0.15
    ).cast(pl.Float32).alias("churn_risk_score"),

])

# Encode Categoricals for ML
ecommerce_feat = ecommerce_feat.with_columns([
    pl.col("login_device").cast(pl.Utf8)
      .cast(pl.Categorical).to_physical().alias("login_device_enc"),
    pl.col("payment_mode").cast(pl.Utf8)
      .cast(pl.Categorical).to_physical().alias("payment_mode_enc"),
    pl.col("gender").cast(pl.Utf8)
      .cast(pl.Categorical).to_physical().alias("gender_enc"),
    pl.col("order_cart").cast(pl.Utf8)
      .cast(pl.Categorical).to_physical().alias("order_cart_enc"),
    pl.col("marital_status").cast(pl.Utf8)
      .cast(pl.Categorical).to_physical().alias("marital_status_enc"),
    pl.col("satisfaction_band").cast(pl.Categorical)
      .to_physical().alias("satisfaction_band_enc"),
    pl.col("tenure_segment").cast(pl.Categorical)
      .to_physical().alias("tenure_segment_enc"),
    pl.col("complain").cast(pl.Int8),
])

# Round cashback columns to whole numbers
ecommerce_feat = ecommerce_feat.with_columns([
    pl.col("cashback_amount", "cashback_per_order").round(0).cast(pl.Int32),
])

# Write to Silver 
ecommerce_feat.write_parquet(f"{SILVER}/ecommerce_features.parquet",
                             compression="snappy")
print(f"\nSilver written: {SILVER}/ecommerce_features.parquet")


Silver written: /lakehouse/default/Files/silver/ecommerce_features.parquet


**STRATIFIED SPLIT AND SAVE**

In [9]:
# STRATIFIED SPLIT AND SAVE
FEATURE_COLS = [
    "tenure", "warehouse_to_home", "hours_on_app",
    "device_registered", "satisfaction_score",
    "number_of_address", "order_amount_hike_from_last_year",
    "coupon_used", "order_count", "day_since_last_order",
    "cashback_amount", "complain",
    # Engineered
    "recency_risk", "cashback_per_order", "coupon_dependency",
    "app_engagement_intensity", "churn_risk_score",
    "high_order_growth",
    # Encoded categoricals
    "login_device_enc", "payment_mode_enc", "gender_enc",
    "order_cart_enc", "marital_status_enc",
    "satisfaction_band_enc", "tenure_segment_enc",
]
LABEL_COL = "churn"

def stratified_split(
    df: pl.DataFrame,
    label_col: str,
    test_size: float = 0.2,
    seed: int = 42
) -> tuple[pl.DataFrame, pl.DataFrame]:
    classes = df[label_col].unique().to_list()
    train_parts, test_parts = [], []
    for cls in classes:
        part = df.filter(pl.col(label_col) == cls)\
                 .sample(fraction=1.0, shuffle=True, seed=seed)
        idx = int(len(part) * (1 - test_size))
        train_parts.append(part[:idx])
        test_parts.append(part[idx:])
    train = pl.concat(train_parts)\
              .sample(fraction=1.0, shuffle=True, seed=seed)
    test  = pl.concat(test_parts)\
              .sample(fraction=1.0, shuffle=True, seed=seed)
    return train, test

# Select only ML-ready columns
df_ml = ecommerce_feat.select(FEATURE_COLS + [LABEL_COL, "customerid"])

# Split
df_train, df_test = stratified_split(df_ml, label_col=LABEL_COL)

# Verify stratification
for name, part in [("Full", df_ml),
                   ("Train", df_train),
                   ("Test", df_test)]:
    rate = part[LABEL_COL].mean()
    print(f"{name:6s}: {len(part):>6,} rows | "
          f"churn rate: {rate:.2%}")

# Save ML-ready Parquet
df_train.write_parquet(f"{SILVER}/train.parquet", compression="snappy")
df_test.write_parquet(f"{SILVER}/test.parquet",   compression="snappy")
df_ml.write_parquet(f"{SILVER}/features_ml.parquet", compression="snappy")

# Verify files
for fname in ["train.parquet", "test.parquet", "features_ml.parquet"]:
    size_mb = os.path.getsize(f"{SILVER}/{fname}") / (1024*1024)

# Convert to NumPy for sklearn
X_train = df_train.select(FEATURE_COLS).to_numpy()
y_train = df_train[LABEL_COL].to_numpy()
X_test  = df_test.select(FEATURE_COLS).to_numpy()
y_test  = df_test[LABEL_COL].to_numpy()
del df_train, df_test


Full  :  5,630 rows | churn rate: 16.84%
Train :  4,503 rows | churn rate: 16.83%
Test  :  1,127 rows | churn rate: 16.86%
